# FlashInspector AI – Train on Kaggle (GPU)

Train the fire safety YOLO model on Kaggle's free GPU.

**Before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 x2).
2. Add your [Roboflow API key](https://app.roboflow.com/settings/api) as a **Secret**: Notebook setup → Add-ons → Secrets → add `ROBOFLOW_API_KEY`, or paste it when prompted below.
3. This notebook clones the repo from GitHub; set `REPO_URL` in the next cell if using a fork.

## 1. Check GPU

In [ ]:
!nvidia-smi 2>/dev/null || echo "No GPU. Enable: Settings → Accelerator → GPU (P100 or T4)"

## 2. Clone repo and go to project

In [ ]:
REPO_URL = "https://github.com/patrisiyarum/fire.git"

!git clone --depth 1 $REPO_URL /kaggle/working/fire
%cd /kaggle/working/fire/flashinspector-ai
import os
print("Project dir:", os.getcwd())

## 3. Install dependencies

You may see dependency conflict warnings; **ignore them** — training will still work.

In [ ]:
!pip install -q --upgrade-strategy only-if-needed ultralytics roboflow opencv-python python-dotenv pyyaml tqdm

## 4. Roboflow API key

**Option A (easiest):** Replace `paste_your_key_here` below with your [Roboflow API key](https://app.roboflow.com/settings/api), then run the cell. **Do not commit or share the notebook with the key in it.**

**Option B:** Use Kaggle Secrets (Add-ons → Secrets, add `ROBOFLOW_API_KEY`); the cell will use it if set.

In [ ]:
import os

# Paste your key here (replace the string) if Secrets aren't working:
ROBOFLOW_API_KEY = "paste_your_key_here"

if os.environ.get("ROBOFLOW_API_KEY"):
    print("Using ROBOFLOW_API_KEY from Secrets.")
elif ROBOFLOW_API_KEY and ROBOFLOW_API_KEY != "paste_your_key_here":
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
    print("Using ROBOFLOW_API_KEY from this cell.")
else:
    raise RuntimeError(
        "Set your key: replace paste_your_key_here above with your Roboflow API key, or add ROBOFLOW_API_KEY in Add-ons → Secrets."
    )

## 5a. Download Roboflow datasets

In [ ]:
!python download_datasets.py

## 5b. Download FireSafetyNet (Zenodo) – All 6 Services

Downloads all [FireSafetyNet](https://zenodo.org/records/13358169) services:

| Service | Task | Classes |
|---------|------|---------|
| S1 | Detection | fire blanket, fire extinguisher, manual call point, smoke detector |
| S2 | Detection | FSE marking signs |
| S3 | Segmentation | fire extinguisher condition (blocked/non-compliant) |
| S4 | Segmentation | fire extinguisher amodal (partially obscured) |
| S5 | Detection | inspection tags |
| S6 | Detection | fire class symbols |

Services 1,2,5,6 → detection model. Services 3,4 → segmentation model. Run before Section 6.

In [ ]:
# Download ALL 6 FireSafetyNet services from Zenodo
!python download_external_datasets.py --all

## 6. Merge datasets

In [ ]:
!python prepare_dataset.py

## 7. Train on Kaggle GPU

Two models to train:

1. **Detection** (`detect`): Uses merged_dataset (S1, S2, S5, S6 + Roboflow datasets). Detects fire blankets, extinguishers, call points, smoke detectors, marking signs, inspection tags, fire class symbols, emergency exits, and more.
2. **Segmentation** (`segment`): Uses merged_segmentation_dataset (S3, S4). Segments fire extinguisher condition and occlusion.

Run the detection cell first, then the segmentation cell.

In [ ]:
# --- Train DETECTION model (S1, S2, S5, S6 + Roboflow) ---
EPOCHS = 50
BATCH = 8
MODEL = "yolov8n.pt"

!python train.py --task detect --epochs {EPOCHS} --batch {BATCH} --model {MODEL}

In [ ]:
# --- Train SEGMENTATION model (S3, S4) ---
SEG_EPOCHS = 50
SEG_BATCH = 8
SEG_MODEL = "yolov8n-seg.pt"

!python train.py --task segment --epochs {SEG_EPOCHS} --batch {SEG_BATCH} --model {SEG_MODEL}

## 8. Save models to Kaggle Output

Copies both detection and segmentation `best.pt` to `/kaggle/working/` so they appear in the **Output** tab for download.

In [ ]:
import shutil
from pathlib import Path

models = {
    "best_detect.pt": Path("runs/fire_safety/weights/best.pt"),
    "best_segment.pt": Path("runs/fire_safety_seg/weights/best.pt"),
}

saved = 0
for dest_name, src in models.items():
    if src.exists():
        dest = Path(f"/kaggle/working/{dest_name}")
        shutil.copy(src, dest)
        print(f"Saved {dest_name} -> {dest}")
        saved += 1
    else:
        print(f"Not found: {src} (skipping {dest_name})")

if saved == 0:
    print("\nNo models found. Check the training output above for errors.")
else:
    print(f"\n{saved} model(s) saved. Download from the Output tab when the run finishes.")